# Import Library

In [70]:
from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.feature_selection import SelectFromModel
from sklearn.base import clone 

# Linear Model 
from sklearn.linear_model import Ridge, LinearRegression, ElasticNet, SGDRegressor
from sklearn.svm import LinearSVR

import re
import joblib
import os 
from scipy import sparse
import numpy as np 
import pandas as pd 

# Load & Split data

In [71]:
data = pd.read_csv('Watches_train_cleaned.csv')
binned = pd.qcut(
    data['price'], 
    q = 10, 
    duplicates='drop', 
)

train_df, valid_df = train_test_split(
    data, 
    test_size=0.1,
    shuffle = True, 
    stratify = binned, 
    random_state = 42,
)

target = 'price'

y_train = train_df[target].copy()
y_valid = valid_df[target].copy()

x_train_raw = train_df.drop (columns=[target]).copy()
x_valid_raw = valid_df.drop (columns=[target]).copy()

# EDA & Feature Overview

In [72]:
missing_summary = pd.DataFrame({
    'missing_count': x_train_raw.isna().sum(), 
    'missing_pct': x_train_raw.isna().mean() * 100, 
    'n_unique': x_train_raw.nunique(dropna = True),
}).sort_values('missing_pct', ascending=False)

missing_summary = missing_summary[missing_summary['missing_count'] > 0]
missing_summary

,missing_count,missing_pct,n_unique
name,47320,26.436197,95977
ref,21329,11.915842,27769
case_size_mm,12796,7.148723,429


Check count of unique values

In [73]:
categorical_cols = ['brand', 'model', 'ref', 'mvmt', 'casem', 'bracem', 'cond']
x_train_raw[categorical_cols].nunique(dropna = True).sort_values (ascending = False) 

ref       27769
model       924
brand        24
mvmt          4
casem         4
bracem        4
cond          3
dtype: int64

Numeric columns overview

In [74]:
numeric_cols = ['yop', 'is_female', 'case_size_mm']
x_train_raw[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
yop,178997.0,2007.341743,39.920980,1550.0,2005.0,2020.0,2023.0,2024.0
is_female,178997.0,0.143343,0.350423,0.0,0.0,0.0,0.0,1.0
case_size_mm,166201.0,38.999763,5.406650,3.0,36.0,40.0,42.0,55.0


Check token of names

In [75]:
name_text = x_train_raw['name'].fillna ('').astype (str)

pd.DataFrame({
    'char_count': name_text.str.len(), 
    'token_count': name_text.str.findall (r"\b\w+\b").str.len(), 
    'digit_count': name_text.str.count (r"\d"),
}).describe().T

,count,mean,std,min,25%,50%,75%,max
char_count,178997.0,41.613921,32.288885,0.0,0.0,41.0,65.0,228.0
token_count,178997.0,6.695408,5.326130,0.0,0.0,7.0,10.0,32.0
digit_count,178997.0,5.261161,5.597783,0.0,0.0,4.0,8.0,72.0


In [76]:
common_tokens = (
    name_text.str.lower().str.findall (r"\b[a-z0-9]+\b").explode().value_counts().head(15)
)
common_tokens

name
omega          28385
chronograph    19464
watch          17809
automatic      17669
dial           16720
cartier        13331
steel          12112
seamaster      11805
breitling      11664
seiko          11535
gold           11433
black          11023
longines       10611
s               9772
heuer           9487
Name: count, dtype: int64

# Experiment Feature sets for Feature Engineering

In [77]:
TfidfVectorizer(
    lowercase=True, 
    ngram_range=(1,2), 
    min_df=5, 
    max_df=0.85, 
    max_features=3000,
    sublinear_tf=True,
)

keywords = [
    "gold", "rose gold", "white gold",
    "yellow gold", "platinum",
    "titanium", "steel", "ceramic",
    "bronze",
    "diamond", "diamonds", "gem",
    "skeleton",
    "limited", "anniversary", "boutique",
    "full set",
    "box", "papers", "card",
    "certificate",
    "chronograph", "perpetual",
    "tourbillon", "moonphase",
    "gmt", "datejust", "daytona",
    "submariner"
]

target = 'price'

# Tabular Feature

In [78]:
tabular_cols = [
    'brand', 'model', 
    'ref', 'mvmt', 'casem', 
    'bracem', 'yop', 'cond', 
    'is_female', 'case_size_mm', 
]

categorical_cols = [
    'brand', 'model', 
    'ref', 'mvmt', 'casem', 
    'bracem', 'cond'
]

numeric_cols = [
    'yop', 'is_female', 
    'case_size_mm',
]

x_train = train_df[tabular_cols].copy()
x_valid = valid_df[tabular_cols].copy()

y_train = np.log1p(train_df[target])
y_valid = np.log1p(valid_df[target]) 

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = 'median')), 
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = 'constant', fill_value='missing')), 
    ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=5))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols), 
    ('cat', categorical_pipeline, categorical_cols)
])

baseline_model = Pipeline([
    ('preprocessor', preprocessor), 
    ('model', Ridge(alpha = 10.0))
])

baseline_model.fit(x_train, y_train)
valid_pred_log = baseline_model.predict (x_valid)
valid_pred = np.expm1 (valid_pred_log)

y_valid_dollars = valid_df[target]

rmse_log = mean_squared_error(y_valid, valid_pred_log) ** 0.5
mae_dollars = mean_absolute_error(y_valid_dollars, valid_pred)
rmse_dollars = mean_squared_error(y_valid_dollars, valid_pred) ** 0.5
r2_log = r2_score (y_valid, valid_pred_log)

print ('Baseline: tabular only')
print (f'RMSE log price: {rmse_log}')
print (f'MAE dollars: {mae_dollars}')
print (f'RMSE dollars: {rmse_dollars}')
print (f'R2 log price: {r2_log}')

Baseline: tabular only
RMSE log price: 0.42659984145014823
MAE dollars: 6000.091370867992
RMSE dollars: 69668.65208804896
R2 log price: 0.8969402893917245


* RMSE dollars is much larger than MAE dollars
* A small number of expensive watches or bad misses are creating large squared errors

# Tabular + TF-IDF Feature

In [79]:
tfidf_cols = tabular_cols + ['name']
x_train_tfidf = train_df[tfidf_cols].copy()
x_valid_tfidf = valid_df[tfidf_cols].copy() 

tfidf_preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols), 
    ('cat', categorical_pipeline, categorical_cols), 
    ('name_tfidf', TfidfVectorizer(
        lowercase=True, 
        ngram_range=(1,2),
        min_df=5, 
        max_df=0.85, 
        max_features=3000,
        sublinear_tf=True, 
    ), 'name')
])

x_train_tfidf['name'] = x_train_tfidf['name'].fillna ("")
x_valid_tfidf['name'] = x_valid_tfidf['name'].fillna ("") 

tfidf_model = Pipeline([
    ('preprocessor', tfidf_preprocessor), 
    ('model', Ridge(alpha = 10.0))
])

In [80]:
tfidf_model.fit (x_train_tfidf, y_train)
valid_pred_log = tfidf_model.predict (x_valid_tfidf)
valid_pred = np.expm1(valid_pred_log)
y_valid_dollars = valid_df[target] 

rmse_log = mean_squared_error(y_valid, valid_pred_log) ** 0.5
mae_dollars = mean_absolute_error(y_valid_dollars, valid_pred)
rmse_dollars = mean_squared_error(y_valid_dollars, valid_pred) ** 0.5
r2_log = r2_score (y_valid, valid_pred_log)

print ('Tabular + TfIDF')
print(f" RMSE log price: {rmse_log}")
print (f"MAE dollars: {mae_dollars}")
print (f"RMSE dollars: {rmse_dollars}")
print (f"R2 log price: {r2_log}")


Tabular + TfIDF
 RMSE log price: 0.3778184793583527
MAE dollars: 5653.137576305543
RMSE dollars: 67662.78320420967
R2 log price: 0.9191623033011009


Conclusion: Combining Tabular and TF-IDF does improve the overall score 

# Tabular + keyword flags

In [81]:
def add_keywords (df, text_col = 'name'): 
    df = df.copy() 
    text = df[text_col].fillna ("").astype (str).str.lower() 
    for keyword in keywords: 
        clean_name = (
            "has_" + 
            keyword.lower().replace (" ", "_").replace ('-', '_')
        )
        pattern = r"\b" + re.escape (keyword.lower()) + r"\b"
        df[clean_name] = text.str.contains (pattern, regex = True).astype (int)

    return df 

x_train_kw = add_keywords (train_df[tabular_cols + ['name']])
x_valid_kw = add_keywords(valid_df[tabular_cols + ['name']])

keyword_cols = [col for col in x_train_kw.columns if col.startswith ("has_")]
keyword_numeric_cols = numeric_cols + keyword_cols
keyword_preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, keyword_numeric_cols), 
    ('cat', categorical_pipeline, categorical_cols), 
])
keyword_model = Pipeline([
    ('preprocessor', keyword_preprocessor), 
    ('model', Ridge(alpha = 10.0))
])

In [82]:
keyword_model.fit (x_train_kw, y_train)
valid_pred_log = keyword_model.predict (x_valid_kw)
valid_pred = np.expm1(valid_pred_log)
y_valid_dollars = valid_df[target]
rmse_log = mean_squared_error(y_valid, valid_pred_log) ** 0.5 
mae_dollars = mean_absolute_error(y_valid_dollars, valid_pred)
rmse_dollars = mean_squared_error(y_valid_dollars, valid_pred) ** 0.5 
r2_log = r2_score (y_valid, valid_pred_log)

print ('Tabular + keyword flags')
print (f'RMSE log: {rmse_log}')
print (f'MAE dollars: {mae_dollars}')
print (f'RMSE dollars: {rmse_dollars}')
print (f'R2 log price: {r2_log}')

Tabular + keyword flags
RMSE log: 0.4170884418844237
MAE dollars: 5920.531642783282
RMSE dollars: 68386.66735917058
R2 log price: 0.9014846625243639


Conclusion: keyword flags does help to improve accuracy but much less siginificantly than TF-IDF

# Tabular + tfIDF + keyword flags

In [83]:
x_train_all = add_keywords (train_df[tabular_cols + ['name']])
x_valid_all = add_keywords(valid_df[tabular_cols + ['name']])
x_train_all['name'] = x_train_all['name'].fillna ("")
x_valid_all['name'] = x_valid_all['name'].fillna ("") 

keyword_cols = [col for col in x_train_all.columns if col.startswith ('has_')]

all_numeric_cols = numeric_cols + keyword_cols
all_preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, all_numeric_cols), 
    ('cat', categorical_pipeline, categorical_cols), 
    ('name_tfidf', TfidfVectorizer(
        lowercase=True, 
        ngram_range=(1,2), 
        min_df=5, 
        max_df=0.85, 
        max_features=3000, 
        sublinear_tf=True
    ), 'name'), 
])

all_model = Pipeline([
    ('preprocessor', all_preprocessor), 
    ('model', Ridge(alpha = 10.0)) 
])

In [84]:
all_model.fit (x_train_all, y_train) 
valid_pred_log = all_model.predict (x_valid_all)
valid_pred = np.expm1(valid_pred_log)
y_valid_dollars = valid_df[target]

rmse_log = mean_squared_error (y_valid, valid_pred_log) ** 0.5
mae_dollars = mean_absolute_error (y_valid_dollars, valid_pred) 
rmse_dollars = mean_squared_error (y_valid_dollars, valid_pred) ** 0.5 
r2_log = r2_score(y_valid, valid_pred_log)
print (f'Tabular + TF-IDF + keyword flags')
print (f'RMSE log: {rmse_log}') 
print (f'MAE dollars: {mae_dollars}')
print (f'RMSE dollars: {rmse_dollars}')
print (f'R2 log: {r2_log}')

Tabular + TF-IDF + keyword flags
RMSE log: 0.3777504271343917
MAE dollars: 5620.300896814127
RMSE dollars: 67224.16319525751
R2 log: 0.9191914214618434


keyword flags seem to add no major improvement to tabular + tfidf

## Inspect Coefficient of features for  tabular + tfidf model 

In [85]:
best_model = tfidf_model
preprocessor = best_model.named_steps['preprocessor']
ridge = best_model.named_steps['model']

feature_names = preprocessor.get_feature_names_out() 
coefs = ridge.coef_

coef_df = pd.DataFrame({ 
    'feature': feature_names, 
    'coef': coefs, 
})

coef_df['abs_coef'] = coef_df['coef'].abs() 
print (coef_df.sort_values ('coef', ascending = False).head (5)); print()
print (coef_df.sort_values ('coef', ascending = True).head (5)); print()

                         feature      coef  abs_coef
10900     name_tfidf__tourbillon  2.978477  2.978477
15      cat__brand_Richard Mille  2.088292  2.088292
9862              name_tfidf__gs  1.649639  1.649639
10675       name_tfidf__seiko gs  1.605764  1.605764
14     cat__brand_Patek Philippe  1.364608  1.364608

                             feature      coef  abs_coef
17                  cat__brand_Seiko -2.206772  2.206772
12                   cat__brand_Oris -1.211147  1.211147
10673  name_tfidf__seiko chronograph -1.082342  1.082342
10416             name_tfidf__pocket -0.969837  0.969837
789                cat__model_Sporto -0.927025  0.927025



In [86]:
tfidf_coef_df = coef_df[coef_df['feature'].str.startswith('name_tfidf__')].copy()
tfidf_coef_df['term'] = tfidf_coef_df['feature'].str.replace ('name_tfidf__', '', regex = False)
print (tfidf_coef_df.sort_values('coef', ascending = False).head (5)); print()
print (tfidf_coef_df.sort_values ('coef', ascending = True).head (5)); print()

                      feature      coef  abs_coef        term
10900  name_tfidf__tourbillon  2.978477  2.978477  tourbillon
9862           name_tfidf__gs  1.649639  1.649639          gs
10675    name_tfidf__seiko gs  1.605764  1.605764    seiko gs
11121    name_tfidf__グランドセイコー  1.359143  1.359143    グランドセイコー
10368   name_tfidf__perpetual  1.115065  1.115065   perpetual

                             feature      coef  abs_coef               term
10673  name_tfidf__seiko chronograph -1.082342  1.082342  seiko chronograph
10416             name_tfidf__pocket -0.969837  0.969837             pocket
11067             name_tfidf__zenith -0.827130  0.827130             zenith
10839         name_tfidf__swiss made -0.796750  0.796750         swiss made
10672    name_tfidf__seiko automatic -0.777844  0.777844    seiko automatic



In [87]:
name_text = x_train_tfidf['name'].fillna ("").astype (str).str.lower() 
important_terms = (tfidf_coef_df.sort_values('coef', ascending = False).head(20)['term'].tolist())
for term in important_terms: 
    count = name_text.str.contains (r"\b" + re.escape(term) + r"\b").sum() 
    print (term, count)

tourbillon 517
gs 192
seiko gs 182
グランドセイコー 83
perpetual 1479
repeater 94
rainbow 103
minute repeater 92
55 751
minute 103
grand complications 224
1704 119
オーバーホール済み 66
skeleton 566
dial 18kt 161
richard mille 461
mille 461
ox 395
openworked 153
richard 499


Analysis: 
* strongest positive text signals are keywords: tourbillon, perpetual, repeater, minute repeater,...
* 1704, 55, ox are some suspicious terms, unclear whether or not they have signals
* negative signals makes sense, including stuffs like: seiko, pocket, seiko chronograph, quartz,...

# Analyze erorr by price bucket

In [89]:
error_df = valid_df.copy() 
error_df['pred_log'] = all_model.predict (x_valid_all) 
error_df['pred_price'] = np.expm1 (error_df['pred_log'])
error_df['signed_error'] = (error_df['price'] - error_df['pred_price'])
error_df['abs_error'] = (error_df['price'] - error_df['pred_price']).abs()
error_df['log_error'] = (np.log1p (error_df['price']) - error_df['pred_log'])
error_df['abs_log_error'] = error_df['log_error'].abs() 
error_df['price_bins'] = pd.qcut (
    error_df['price'], 
    q = 10, 
    duplicates='drop', 
)

error_df['price_bins'] = pd.qcut (
    error_df['price'], 
    q = 10, 
    duplicates='drop'
)

error_df.groupby('price_bins').agg (
    count = ('price', 'size'),
    mae = ('abs_error', 'mean'), 
    median_abs_error = ('abs_error', 'median'),
    rmse = ('abs_error', lambda x: np.sqrt (np.mean (x**2))), 
    mean_abs_log_error = ('abs_log_error', 'mean'), 
)

,count,mae,median_abs_error,rmse,mean_abs_log_error
price_bins,,,,,
"(71.999, 1449.8]",1989,365.974120,218.354126,582.894240,0.382524
"(1449.8, 2639.6]",1989,610.488544,400.817043,1110.283101,0.263273
"(2639.6, 3999.0]",1990,1031.229324,624.566888,5336.972485,0.252791
"(3999.0, 5495.0]",1991,1075.829733,721.452926,1584.159025,0.218700
"(5495.0, 7344.0]",1988,1334.575688,978.726209,1936.923723,0.204807
"(7344.0, 9984.0]",1986,1782.106908,1246.469286,2693.667731,0.207452
"(9984.0, 13975.0]",1990,2491.125629,1688.941845,3818.129709,0.216265
"(13975.0, 20362.0]",1988,3336.382586,2380.197901,4698.565514,0.217281
"(20362.0, 41999.2]",1989,7046.177395,4905.062648,10878.658645,0.276909


Analysis: 
* lowets mae: 365
* middle mae: 1k - 3.3k
* highest mae: 37.1k
* second highest mae: 7.04k
* highest rmse: 212k
* lowest abs log error: 0.382
* middle buckets log error: 0.26 - 0.21
* highest buckets log error: 0.319

Conclusion: 
* The model is good in the middle of the market
* The top price bucket domiates dollar error
* The model struggles at both extremes, very cheap and very expensive watches

Strategy:
* Experiements with non-linear models 
* Some high-end indicator features should be added
* Consider having interactions features: brand + model, brand + high-end-kw,...
* Training on log price, evaluating per bucket might help 
* overprediction and underprediction should be checked carefully in top buckets

# Experiment coefficient for Ridge regularization

In [90]:
MODEL_DIR = 'Model/linear_model_selection'
os.makedirs (MODEL_DIR, exist_ok=True)

alphas = [0.1, 0.3, 1, 3, 10, 30, 100, 300] 
ridge_results = []

for alpha in alphas: 
    name = f'ridge_{alpha}_experiment'

    if os.path.exists(f'{MODEL_DIR}/{name}.joblib'):
        print (f'Loading model {name}...')
        model = joblib.load (f'{MODEL_DIR}/{name}.joblib')
    else: 
        print (f'Fitting model {name}...', end = ' ')
        model = Pipeline([
            ('preprocessor', tfidf_preprocessor), 
            ('model', Ridge(alpha = alpha)) 
        ])
        model.fit (x_train_tfidf, y_train) 
        joblib.dump (model, f'{MODEL_DIR}/{name}.joblib')
        print (f'SAVED!')

    valid_pred_log = model.predict (x_valid_tfidf)
    valid_pred = np.expm1(valid_pred_log)
    rmse_log = mean_squared_error(y_valid, valid_pred_log) ** 0.5
    mae_dollars = mean_absolute_error(valid_df[target], valid_pred)
    rmse_dollars = mean_squared_error(valid_df[target], valid_pred)  ** 0.5 
    r2_log = r2_score(y_valid, valid_pred_log) 

    ridge_results.append({
        'alpha': alpha, 
        'rmse_log': rmse_log, 
        'mae_dollars': mae_dollars, 
        'rmse_dollars': rmse_dollars, 
        'r2_log': r2_log, 
    })

ridge_results_df = pd.DataFrame (ridge_results).sort_values('rmse_log')
ridge_results_df

Fitting model ridge_0.1_experiment... SAVED!
Fitting model ridge_0.3_experiment... SAVED!
Fitting model ridge_1_experiment... SAVED!
Fitting model ridge_3_experiment... SAVED!
Fitting model ridge_10_experiment... SAVED!
Fitting model ridge_30_experiment... SAVED!
Fitting model ridge_100_experiment... SAVED!
Fitting model ridge_300_experiment... SAVED!


,alpha,rmse_log,mae_dollars,rmse_dollars,r2_log
2,1.0,0.363426,5325.567246,66540.366795,0.925204
1,0.3,0.363759,5355.159382,66506.640993,0.925067
0,0.1,0.364288,5371.546053,66509.983307,0.924848
3,3.0,0.366049,5386.967287,66784.551569,0.924120
4,10.0,0.377818,5653.137576,67662.783204,0.919162
5,30.0,0.399974,6146.266377,69151.089604,0.909404
6,100.0,0.437029,6836.735045,71061.915161,0.891840
7,300.0,0.481951,7679.146103,72733.788876,0.868461


alpha  = 1.0 has the lowest error

In [91]:
best_alpha = ridge_results_df['alpha'].iloc[0]
ridge_tfidf_model = Pipeline([
    ('preprocessor', tfidf_preprocessor), 
    ('model', Ridge(alpha = best_alpha)), 
])

best_error_df = valid_df.copy() 
ridge_tfidf_model.fit (x_train_tfidf, y_train)
best_error_df['pred_log'] = ridge_tfidf_model.predict (x_valid_tfidf) 
best_error_df['pred_price'] = np.expm1 (best_error_df['pred_log'])
best_error_df['signed_error'] = (best_error_df['price'] - best_error_df['pred_price'])
best_error_df['abs_error'] = (best_error_df['price'] - best_error_df['pred_price']).abs()
best_error_df['log_error'] = (np.log1p (best_error_df['price']) - best_error_df['pred_log'])
best_error_df['abs_log_error'] = best_error_df['log_error'].abs() 
best_error_df['price_bins'] = pd.qcut (
    best_error_df['price'], 
    q = 10, 
    duplicates='drop', 
)

best_bucket_metrics = best_error_df.groupby ('price_bins').agg (
    count = ('price', 'size'), 
    mae = ('abs_error', 'mean'),
    median_signed_error = ('signed_error', 'mean'), 
    median_abs_error = ('abs_error', 'median'), 
    rmse = ('abs_error', lambda x: np.sqrt (np.mean (x ** 2))), 
    mean_abs_log_error = ('abs_log_error', 'mean')
)

best_bucket_metrics

,count,mae,median_signed_error,median_abs_error,rmse,mean_abs_log_error
price_bins,,,,,,
"(71.999, 1449.8]",1989,341.442595,-247.213106,202.548944,554.659730,0.361617
"(1449.8, 2639.6]",1989,575.466468,-247.286121,365.597181,1044.307903,0.252839
"(2639.6, 3999.0]",1990,981.188580,-447.494612,581.873939,4702.315381,0.245059
"(3999.0, 5495.0]",1991,1032.297859,-295.403941,686.976087,1562.796310,0.211288
"(5495.0, 7344.0]",1988,1273.329646,-275.336758,916.458684,1874.107134,0.195903
"(7344.0, 9984.0]",1986,1762.834902,-256.959168,1217.480370,2645.306351,0.205077
"(9984.0, 13975.0]",1990,2371.335183,-242.986556,1537.579839,3711.611949,0.204179
"(13975.0, 20362.0]",1988,3211.558085,579.336248,2257.797252,4665.167821,0.206728
"(20362.0, 41999.2]",1989,6668.839449,768.250015,4331.697664,10729.410157,0.257148


Though alpha = 1 does improve the model, the noise created by top priced watches is still significant, upto 32k of MAE error for the top bucket

Using mixed encoding does improve the error a little bit compared to purely Ridge regression with one-hot encoding 

In [92]:
feature_names = ridge_tfidf_model.named_steps['preprocessor'].get_feature_names_out()
coefs = ridge_tfidf_model.named_steps['model'].coef_

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coef': coefs, 
    'abs_coef': np.abs (coefs), 
}).sort_values ('abs_coef', ascending=False)
coef_df.head (5)

,feature,coef,abs_coef
10900,name_tfidf__tourbillon,3.537595,3.537595
5123,cat__ref_95.9000.8812/78.M9000,2.571764,2.571764
4545,cat__ref_5711/1A-018,2.479461,2.479461
15,cat__brand_Richard Mille,2.433406,2.433406
5124,cat__ref_95.9000.8812/78.R584,2.326565,2.326565


In [93]:
coef_df.sort_values('coef', ascending=False).head (5)

,feature,coef,abs_coef
10900,name_tfidf__tourbillon,3.537595,3.537595
5123,cat__ref_95.9000.8812/78.M9000,2.571764,2.571764
4545,cat__ref_5711/1A-018,2.479461,2.479461
15,cat__brand_Richard Mille,2.433406,2.433406
5124,cat__ref_95.9000.8812/78.R584,2.326565,2.326565


In [94]:
coef_df.sort_values('coef', ascending=False).head (5)


,feature,coef,abs_coef
10900,name_tfidf__tourbillon,3.537595,3.537595
5123,cat__ref_95.9000.8812/78.M9000,2.571764,2.571764
4545,cat__ref_5711/1A-018,2.479461,2.479461
15,cat__brand_Richard Mille,2.433406,2.433406
5124,cat__ref_95.9000.8812/78.R584,2.326565,2.326565


* kw like 'tourbilon' carries good signal and can be beneficial for prediction
* brands like richard mille and seiko being flagged is great 
* ref like 8812 affecting the price is something worth considering, this can lead to overfitting

In [95]:
def get_feature_family (feature): 
    if feature.startswith ('name_tfidf__'): 
        return 'tfidf'
    if feature.startswith ('cat__brand_'): 
        return 'brand'
    if feature.startswith ('cat__model_'): 
        return 'model'
    if feature.startswith ('cat__ref_'): 
        return 'ref'    
    if feature.startswith ('mvmt'): 
        return 'movement'    
    if feature.startswith ('num__'): 
        return 'numeric'    
    return 'other' 

coef_df['family'] = coef_df['feature'].apply (get_feature_family) 
family_summary = (
    coef_df.groupby('family').agg (
        n_features = ('feature', 'count'), 
        mean_abs_coef = ('abs_coef', 'mean'), 
        median_abs_coef = ('abs_coef', 'median'), 
        max_abs_coef = ('abs_coef', 'max'), 
    ).sort_values ('max_abs_coef', ascending= False) 
)

family_summary 

,n_features,mean_abs_coef,median_abs_coef,max_abs_coef
family,,,,
tfidf,3000,0.239349,0.174668,3.537595
ref,7351,0.225029,0.167449,2.571764
brand,21,0.782288,0.610448,2.433406
model,850,0.284499,0.202873,1.683278
other,15,0.118578,0.110523,0.294974
numeric,3,0.035320,0.011962,0.082446


In [96]:
coef_df.head(100)['family'].value_counts() 

family
ref      54
tfidf    26
model    16
brand     4
Name: count, dtype: int64

ref is dominating the strongest features, causing high risk of overfitting for the model 

# Experiment removing ref

In [97]:
def add_ref_features(df): 
    df = df.copy() 
    ref_text = df['ref'].fillna ('').astype (str) 
    df['ref_missing'] = ref_text.eq ('').astype (int)
    df['ref_len'] = ref_text.str.len() 
    df['ref_digit_count'] = ref_text.str.count (r"\d")
    df['ref_alpha_count'] = ref_text.str.count (r"[A-Za-z]")
    df['ref_sep_count'] = ref_text.str.count (r"[/.-]")

    return df

x_train_no_ref = add_ref_features(x_train_all) 
x_valid_no_ref = add_ref_features(x_valid_all)

In [98]:
no_ref_numeric_cols = all_numeric_cols + [
    'ref_missing', 
    'ref_len', 
    'ref_digit_count', 
    'ref_alpha_count', 
    'ref_sep_count', 
]

no_ref_categorical_cols = [
    'brand', 
    'model', 
    'mvmt', 
]

no_ref_preprocessor = ColumnTransformer ([
    ('num', numeric_pipeline, no_ref_numeric_cols), 
    ('cat', categorical_pipeline, no_ref_categorical_cols), 
    ('name_tfidf', TfidfVectorizer (
        lowercase=True, 
        ngram_range=(1,2), 
        min_df=5, 
        max_df=0.85, 
        max_features=3000, 
        sublinear_tf=True,
    ), 'name') 
])

no_ref_model = Pipeline([
    ('preprocessor', no_ref_preprocessor), 
    ('model', Ridge(alpha = best_alpha)) 
])

In [99]:
no_ref_model.fit(x_train_no_ref, y_train)
valid_pred_log = no_ref_model.predict (x_valid_no_ref) 
valid_pred = np.expm1(valid_pred_log)
rmse_log = mean_squared_error(y_valid, valid_pred_log) ** 0.5 
mae_dollars = mean_absolute_error(valid_df[target], valid_pred) 
rmse_dollars = mean_squared_error(valid_df[target], valid_pred) ** 0.5 
r2_log = r2_score (y_valid, valid_pred_log) 

print (f'Ridge + TF-IDF without ref one-hot') 
print (f'RMSE log: {rmse_log}') 
print (f'MAE dollars: {mae_dollars}') 
print (f'RMSE dollars: {rmse_dollars}') 
print (f'R2 log: {r2_log}') 

Ridge + TF-IDF without ref one-hot
RMSE log: 0.43773034196345156
MAE dollars: 7017.674109294264
RMSE dollars: 67894.31363714115
R2 log: 0.8914922290065479


After removing one-hot ref, the performance is worsened slightly, meaning this feature does help the model to a certain extent

In [100]:
feature_names = no_ref_model.named_steps['preprocessor'].get_feature_names_out()
coefs = no_ref_model.named_steps['model'].coef_
no_ref_coef_df = pd.DataFrame ({
    'feature': feature_names,
    'coef': coefs, 
    'abs_coef': np.abs (coefs), 
}).sort_values('abs_coef', ascending = False) 
no_ref_coef_df.head (6)

,feature,coef,abs_coef
49,cat__brand_Richard Mille,2.626095,2.626095
3572,name_tfidf__tourbillon,2.412249,2.412249
51,cat__brand_Seiko,-2.251132,2.251132
1427,name_tfidf__424 13,-2.202388,2.202388
3578,name_tfidf__travel time,1.955283,1.955283
1426,name_tfidf__424 10,-1.942378,1.942378


Among the top 6 features, we see there are brands, models, and names

# Experiment removing model 

In [101]:
no_ref_model_numeric_cols = no_ref_numeric_cols + [
    'model_count', 
    'model_freq', 
]

model_counts = x_train_no_ref['model'].value_counts() 
model_freq = x_train_no_ref['model'].value_counts(normalize = True)

for df in [x_train_no_ref, x_valid_no_ref]: 
    df['model_count'] = df['model'].map (model_counts).fillna (0)
    df['model_freq'] = df['model'].map (model_freq).fillna (0)

no_ref_no_model_categorical_cols = ['brand', 'mvmt'] 

no_ref_no_model_preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, no_ref_model_numeric_cols), 
    ('cat', categorical_pipeline, no_ref_no_model_categorical_cols), 
    ('name_tfidf', TfidfVectorizer(
        lowercase=True, 
        ngram_range=(1,2), 
        min_df=5, 
        max_df=0.85, 
        max_features=3000, 
        sublinear_tf=True, 
    ), 'name') 
])

no_ref_no_model_model = Pipeline ([
    ('preprocessor', no_ref_no_model_preprocessor), 
    ('model', Ridge(alpha = best_alpha)), 
])

no_ref_no_model_model.fit (x_train_no_ref, y_train) 
valid_pred_log = no_ref_no_model_model.predict (x_valid_no_ref) 
valid_pred = np.expm1 (valid_pred_log) 
rmse_log = mean_squared_error (y_valid, valid_pred_log) ** 0.5 
mae_dollars = mean_absolute_error (valid_df[target], valid_pred) 
rmse_dollars = mean_squared_error (valid_df[target], valid_pred) ** 0.5 
r2_log = r2_score (y_valid, valid_pred_log) 

print ('Ridge + TF-IDF without ref & model one hot') 
print (f'RMSE log: {rmse_log}') 
print (f'MAE dollars: {mae_dollars}') 
print (f'RMSE dollars: {rmse_dollars}') 
print (f'R2 log: {r2_log}') 

Ridge + TF-IDF without ref & model one hot
RMSE log: 0.5288387291435721
MAE dollars: 8288.243482195814
RMSE dollars: 68654.67931674868
R2 log: 0.8416222951567129


Without one-hot encoded model, the error increases noticably


# Experiment different min frequency 

In [102]:
min_freq_results = [] 
fitted_min_frequency_pipelines = {}
for min_freq in [5, 10, 20, 50]: 

    model_name = f"min_freq_{min_freq}"
    cat_pipeline = Pipeline ([
        ('imputer', SimpleImputer (strategy = 'constant', fill_value = 'missing')), 
        ('onehot', OneHotEncoder (handle_unknown = 'ignore', min_frequency = min_freq)), 
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, no_ref_numeric_cols),
        ("cat", cat_pipeline, no_ref_categorical_cols),
        ("name_tfidf", TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=5,
            max_df=0.85,
            max_features=3000,
            sublinear_tf=True,
        ), "name")
    ])

    if os.path.exists(f'{MODEL_DIR}/{model_name}.joblib'):
        print (f'Loading model {model_name}...', end =' ')
        model = joblib.load (f'{MODEL_DIR}/{model_name}.joblib')
        print ('DONE!')
    else: 
        print (f'Fitting model {model_name}...', end = ' ')
        model = Pipeline([
            ("preprocessor", preprocessor),
            ("model", Ridge(alpha=best_alpha)),
        ])
        
        model.fit(x_train_no_ref, y_train)
        joblib.dump (model, f'{MODEL_DIR}/{model_name}.joblib')
        print ('SAVED!')

    valid_pred_log = model.predict(x_valid_no_ref)
    valid_pred = np.expm1(valid_pred_log)

    min_freq_results.append({
        "min_frequency": min_freq,
        "n_features": len(model.named_steps["preprocessor"].get_feature_names_out()),
        "rmse_log": mean_squared_error(y_valid, valid_pred_log) ** 0.5,
        "mae_dollars": mean_absolute_error(valid_df[target], valid_pred),
        "rmse_dollars": mean_squared_error(valid_df[target], valid_pred) ** 0.5,
        "r2_log": r2_score(y_valid, valid_pred_log),
    })

min_freq_results_df = pd.DataFrame(min_freq_results).sort_values("rmse_log")
min_freq_results_df    

Fitting model min_freq_5... SAVED!
Fitting model min_freq_10... SAVED!
Fitting model min_freq_20... SAVED!
Fitting model min_freq_50... SAVED!


,min_frequency,n_features,rmse_log,mae_dollars,rmse_dollars,r2_log
0,5,3912,0.437730,7017.674109,67894.313637,0.891492
1,10,3824,0.438671,7034.461345,67767.545349,0.891026
2,20,3689,0.440497,7078.303579,67956.379251,0.890116
3,50,3500,0.445025,7181.634078,68260.670634,0.887845


For onehot-encoded features, min_frequency = 10 returns the best result 

# Error Analysis

In [103]:
error_df = valid_df.copy()
error_df["pred_log"] = best_model.predict(x_valid_tfidf)
error_df["pred_price"] = np.expm1(error_df["pred_log"])
error_df["abs_error"] = (error_df["price"] -
error_df["pred_price"]).abs()
error_df["abs_pct_error"] = error_df["abs_error"] / error_df["price"]
error_df["abs_log_error"] = (np.log1p(error_df["price"]) -
error_df["pred_log"]).abs()

error_df["price_bin"] = pd.qcut(error_df["price"], q=10,
duplicates="drop")

error_df.groupby("price_bin").agg(
    count=("price", "size"),
    median_price=("price", "median"),
    mae=("abs_error", "mean"),
    median_abs_error=("abs_error", "median"),
    median_abs_pct_error=("abs_pct_error", "median"),
    mean_abs_log_error=("abs_log_error", "mean"),
)

,count,median_price,mae,median_abs_error,median_abs_pct_error,mean_abs_log_error
price_bin,,,,,,
"(71.999, 1449.8]",1989,775.0,366.420415,219.341360,0.315541,0.382561
"(1449.8, 2639.6]",1989,2025.0,610.765176,399.040199,0.200994,0.263537
"(2639.6, 3999.0]",1990,3300.0,1030.202773,618.484166,0.189486,0.252574
"(3999.0, 5495.0]",1991,4774.0,1075.933363,718.785205,0.154785,0.218548
"(5495.0, 7344.0]",1988,6320.5,1339.426837,984.932179,0.157496,0.205372
"(7344.0, 9984.0]",1986,8500.0,1784.220334,1253.773103,0.146642,0.207419
"(9984.0, 13975.0]",1990,11750.0,2500.303008,1694.883258,0.145077,0.216737
"(13975.0, 20362.0]",1988,16595.0,3348.481558,2381.650613,0.143038,0.217683
"(20362.0, 41999.2]",1989,27085.0,7048.693063,4904.756246,0.175728,0.276491


# Experiment removing bracem and casem

In [104]:
ablation_results = []
best_min_frequency = 5
best_alpha = 1.0

def run_ridge_ablation(label, 
                       categorical_cols_used, 
                       numeric_cols_used):
    
    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="constant",
        fill_value="missing")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=best_min_frequency
        )),
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_cols_used),
        ("cat", cat_pipeline, categorical_cols_used),
        ("name_tfidf", TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=5,
            max_df=0.85,
            max_features=3000,
            sublinear_tf=True,
        ), "name")
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=best_alpha)),
    ])

    model.fit(x_train_tfidf, y_train)

    valid_pred_log = model.predict(x_valid_tfidf)
    valid_pred = np.expm1(valid_pred_log)

    feature_names = pd.Series(
        model.named_steps["preprocessor"].get_feature_names_out()
    )

    return {
        "experiment": label,
        "n_features": len(feature_names),
        "rmse_log": mean_squared_error(y_valid,
        valid_pred_log) ** 0.5,
        "mae_dollars": mean_absolute_error(valid_df["price"],
        valid_pred),
        "rmse_dollars": mean_squared_error(valid_df["price"],
        valid_pred) ** 0.5,
        "r2_log": r2_score(y_valid, valid_pred_log),
    }

ablation_results.append(
    run_ridge_ablation(
        "full_current",
        ["brand", "model", "ref", "mvmt", "casem",
        "bracem", "cond"],
        ["yop", "is_female", "case_size_mm"],
    )
)

ablation_results.append(
    run_ridge_ablation(
        "drop_casem_bracem",
        ["brand", "model", "ref", "mvmt", "cond"],
        ["yop", "is_female", "case_size_mm"],
    )
)

ablation_results.append(
    run_ridge_ablation(
        "drop_casem_bracem_cond",
        ["brand", "model", "ref", "mvmt"],
        ["yop", "is_female", "case_size_mm"],
    )
)

ablation_results_df = (
    pd.DataFrame(ablation_results)
    .sort_values("rmse_log")
    .reset_index(drop=True)
)

ablation_results_df

,experiment,n_features,rmse_log,mae_dollars,rmse_dollars,r2_log
0,full_current,11240,0.363426,5325.567246,66540.366795,0.925204
1,drop_casem_bracem,11232,0.370987,5395.066429,66532.184537,0.922059
2,drop_casem_bracem_cond,11229,0.374509,5417.780817,66672.864725,0.920572


# Investigate error on outliers

In [105]:
error_df["signed_error"] = error_df["price"] - error_df["pred_price"]
error_df["signed_pct_error"] = error_df["signed_error"] / error_df["price"]
error_df['price_bins'] = pd.qcut (
    error_df['price'], 
    q = 5, 
    duplicates='drop', 
)

tail_bias = error_df.groupby("price_bins").agg(
    count=("price", "size"),
    median_price=("price", "median"),
    mean_signed_error=("signed_error", "mean"),
    median_signed_error=("signed_error", "median"),
    mean_signed_pct_error=("signed_pct_error", "mean"),
    median_signed_pct_error=("signed_pct_error", "median"),
    mae=("abs_error", "mean"),
    median_abs_pct_error=("abs_pct_error", "median"),
)

tail_bias

,count,median_price,mean_signed_error,median_signed_error,mean_signed_pct_error,median_signed_pct_error,mae,median_abs_pct_error
price_bins,,,,,,,,
"(71.999, 2639.6]",3978,1449.5,-287.308163,-140.891624,-0.305633,-0.140910,488.592795,0.245835
"(2639.6, 5495.0]",3981,4000.0,-440.258060,-210.821517,-0.120074,-0.055169,1053.073812,0.170289
"(5495.0, 9984.0]",3974,7344.0,-304.245476,-73.878911,-0.042812,-0.010169,1561.711660,0.152418
"(9984.0, 20362.0]",3978,13975.0,274.240597,375.696445,0.013109,0.027233,2924.179066,0.144100
"(20362.0, 8778828.0]",3978,41999.5,12285.989864,2238.469715,0.069200,0.056693,22237.485624,0.180811


* Low-price watches have negative signed error, meaning they are overpredicted
* high-price watches have positive signed error, meaning they are underpredicted 
* Model is focusing more in the middle 

# Model Experiment

In [106]:
NUMERIC_COLS = ['yop', 'is_female', 'case_size_mm']
CATEGORICAL_COLS = ['brand', 'model', 'ref', 'mvmt', 'casem', 'bracem', 'cond']
FEATURE_COLS = CATEGORICAL_COLS + NUMERIC_COLS + ['name']

In [107]:
def build_feature_preprocessor (
    onehot_min_frequency = 5, 
):
    numeric_pipeline = Pipeline (
        steps = [
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler',StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps = [
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')), 
            ('onehot', OneHotEncoder(
                    handle_unknown='ignore',
                    min_frequency=onehot_min_frequency,
            )),
        ]
    )

    text_vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1,2),
        min_df=5,
        max_df=0.85,
        max_features=3000,
        sublinear_tf=True,
    )

    preprocessor = ColumnTransformer(
        transformers = [
            ('num', numeric_pipeline, NUMERIC_COLS),
            ('cat', categorical_pipeline, CATEGORICAL_COLS), 
            ('name_tfidf', text_vectorizer, 'name'),
        ]
    )

    return preprocessor

In [108]:
x_train = train_df[FEATURE_COLS].copy() 
x_valid = valid_df[FEATURE_COLS].copy() 

x_train['name'] = x_train['name'].fillna ('').astype (str)
x_valid['name'] = x_valid['name'].fillna ('').astype (str)

preprocessor = build_feature_preprocessor(
    onehot_min_frequency=5,
)

x_train_tfidf = preprocessor.fit_transform (x_train)
x_valid_tfidf = preprocessor.transform (x_valid)

In [ ]:
def evaluate_linear_matrix_model (model, x_valid, y_valid_log, y_valid_price): 
    pred_log = model.predict (x_valid)
    pred_price = np.expm1(pred_log)

    return {
        'rmse_log': mean_squared_error (y_valid_log, pred_log) ** 0.5, 
        'mae_dollars': mean_absolute_error(y_valid_price, pred_price), 
        'rmse_dollars': mean_squared_error (y_valid_price, pred_price) ** 0.5, 
        'r2_log': r2_score (y_valid_log, pred_log),
    }

MODEL_DIR = 'Model/linear_model_selection' 
os.makedirs(MODEL_DIR, exist_ok=True) 
linear_model_candidates = {
    'ridge': Ridge(alpha = 1.0),
    'elastic': ElasticNet (
        alpha = 0.0001, 
        l1_ratio=0.05,
        max_iter=2000,
        random_state=42, 
    ), 

    'sgd_huber': SGDRegressor (
        loss = 'huber', 
        penalty='l2', 
        alpha = 0.00001,
        max_iter = 2000, 
        random_state=42,
    ),

    'linear_svr': LinearSVR (
        C = 1.0,
        epsilon=0.0,
        random_state=42,
        max_iter=5000,
    ),
}

In [110]:
linear_results = []
fitted_linear_models = {}
x_train_linear_raw = train_df[FEATURE_COLS].copy()
x_valid_linear_raw = valid_df[FEATURE_COLS].copy()
x_train_linear_raw['name'] = x_train_linear_raw['name'].fillna ('').astype (str)
x_valid_linear_raw['name'] = x_valid_linear_raw['name'].fillna ('').astype (str)

for model_name, model in linear_model_candidates.items():
    model_path = os.path.join (MODEL_DIR, f"{model_name}_linear_pipeline.joblib")

    if os.path.exists (model_path): 
        print (f'Loading {model_name}...', end = ' ')
        fitted_pipeline = joblib.load (model_path)
        print ('DONE!')
    else: 
        print (f'Fitting {model_name}...', end = ' ')
        fitted_pipeline = Pipeline([
            ('preprocessor', build_feature_preprocessor(
                onehot_min_frequency=5,
            )),
            ('model', clone (model)),
        ])
        fitted_pipeline.fit (x_train_linear_raw, y_train)
        joblib.dump (fitted_pipeline, model_path)
        print ('SAVED!')

    metrics = evaluate_linear_matrix_model(
        fitted_pipeline,
        x_valid_linear_raw,
        y_valid,
        valid_df['price'], 
    )

    fitted_linear_models[model_name] = fitted_pipeline

    linear_results.append ({
        'model': model_name,
        **metrics,
        'model_path': model_path, 
    })

    linear_results_df = (
        pd.DataFrame(linear_results).sort_values('rmse_log').reset_index(drop = True)
    )

linear_results_df

Fitting ridge... SAVED!
Fitting elastic... SAVED!
Fitting sgd_huber... SAVED!
Fitting linear_svr... 

c:\Users\alexh\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


SAVED!


,model,rmse_log,mae_dollars,rmse_dollars,r2_log,model_path
0,ridge,0.363426,5325.567246,66540.366795,0.925204,Model/linear_model_selection\ridge_linear_pipe...
1,linear_svr,0.371310,5125.922769,66912.144513,0.921923,Model/linear_model_selection\linear_svr_linear...
2,elastic,0.392681,5988.072376,68471.100437,0.912677,Model/linear_model_selection\elastic_linear_pi...
3,sgd_huber,0.586576,9124.135518,75475.671649,0.805152,Model/linear_model_selection\sgd_huber_linear_...


Ridge and Linear_SVR are clearly the two most promising linear models